### Self-Attention with Trainable Weights 

In [1]:
import torch


#already tokenized and embedded 
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)
#select word "journey" to calculate attention scores related to it
x_2 = inputs[1]
d_in = inputs.shape[1] #d= 3
d_out= 2 #d=2
#different input and output dimensions

In [2]:
import torch.nn as nn

torch.manual_seed(123)
#defineing random weight matrices for query, key, value projections 
W_query = nn.Parameter(torch.rand(d_in, d_out), requires_grad = False)
W_key = nn.Parameter(torch.rand(d_in, d_out), requires_grad = False)
W_value = nn.Parameter(torch.rand(d_in, d_out), requires_grad = False)



In [3]:
#matrix multiplications (dot product) of input vector against every column in the weight matrix
query_2 = x_2 @ W_query
keys = x_2 @ W_key
values = x_2 @ W_value
print (query_2) #2 dimensional query vector (specified by d_out)

tensor([0.4306, 1.4551])


once we have gotten our query vector we would still need the rest of the input token's keys and values in computing the final context vector with respect to the query vector for the rest of the attention weights 

In [4]:
keys = inputs @ W_key
values = inputs @ W_value
print("keys.shape: ", keys.shape)
print("values.shape: ", values.shape)
#[6,2] 6 inputs each with 2 dimensional embeddings for keys and values 


keys.shape:  torch.Size([6, 2])
values.shape:  torch.Size([6, 2])


Generating the attention score for w22 (journey with respect to query journey)

In [5]:
keys_2 = keys[1]
attn_scores_22 = query_2.dot(keys_2)
print(attn_scores_22) #unormalized attention score

tensor(1.8524)


Generate all with respection to query token 2 (journey)

In [6]:
#matrix multiplication of query vector against keys of all input token(transposed) 
#also same as dot product of query vector against each key vector (for all input tokens)
attn_scores_2 = query_2 @ keys.T
print(attn_scores_2)

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])


we need to now compete the attention weight (attenion score normalized) however unlike before and inputing it directly to softmax we would first have to scale the attention score. (why is explained in the "Self-attention with trainable weights" md file) 

In [7]:
#scale attention score by dividing  by sqrt of embdding dimensions
d_k = keys.shape[-1] #d_k = 2
attn_weights_2 = torch.softmax(attn_scores_2 / torch.sqrt(torch.tensor(d_k)), dim=-1)
print(attn_weights_2) #attention weights for each input token with respect to the query token "journey"

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])


with the attention weights you can now compute a weighted sum with the value vector to get the context vector

In [8]:
context_vec_2 = attn_weights_2 @ values
print(context_vec_2) #context vector for the query token "journey" which is a weighted sum of value vectors of all input tokens

tensor([0.3061, 0.8210])


### Using self-attention class V1

In [9]:
from self_attention_classes import SelfAttention_v1

In [10]:
torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)
print(sa_v1(inputs))
#you can see it matches the context vector from the previous implementation, 

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


### Using self-attention class V2
nn.linear

In [11]:
from self_attention_classes import SelfAttention_v2

torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)
